# Exercises XP: Vector Databases and RAG
Use this guided notebook and fill each TODO before running cells.

## What you'll learn
- Vector search strategies (KNN, ANN) and evaluation.
- Vector database utility (similarity search, RAG).
- Differences between vector DBs, libraries, and plugins.
- Best practices for vector store usage and performance.
- How LMs use context; embedding generation and storage.
- Querying vector stores and applying LMs for QA with retrieved context.

## What you'll build
A functional RAG pipeline with FAISS and ChromaDB, plus QA over retrieved context using a Hugging Face model.

## 0. Setup
Run the install cell once. If your platform needs system deps (e.g., libomp for FAISS), follow instructions in comments.

In [1]:
%pip uninstall -y pydantic-core pydantic
%pip install -U "pydantic<2"
%pip install -U "faiss-cpu>=1.8.0" "chromadb==0.3.21"
%pip install -U "numpy<2" sentence-transformers transformers

Found existing installation: pydantic_core 2.46.4
Uninstalling pydantic_core-2.46.4:
  Successfully uninstalled pydantic_core-2.46.4
Found existing installation: pydantic 2.13.4
Uninstalling pydantic-2.13.4:
  Successfully uninstalled pydantic-2.13.4
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.1/155.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 40.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pyiceberg 0.11.1 requires pydantic!=2.12.0,!=2.12.1,!=2.4.0,!=2.4.1,<3.0,>=2.0, but you have pydantic 1.10.26 which is incompatible.
google-genai 2.10.0 requires pydantic<3.0.0,>=2.12.5, but you have pydantic 1.10.26 which is incompatible.
langchain-core 1.4.8 requires pydantic<3.0.0,>=2.7.4, but you have pydantic 1.10.26 which is incompatible.
albumentations 2.0.8 requires pydantic>=2.9.2, but you have pydantic 

In [1]:
import os
import json
from pathlib import Path

# Fix PydanticImportError and Protobuf VersionError.
# This error occurs because chromadb (or its deps like opentelemetry)
# expects a newer protobuf runtime than what might be currently loaded or available.
# We will aggressively uninstall and then reinstall critical packages in a specific order.

# 1. Aggressively uninstall all potentially conflicting packages
%pip uninstall -y pydantic pydantic-settings chromadb numpy pydantic_core protobuf transformers sentence-transformers faiss-cpu

# 2. Install protobuf first, ensuring the required version (or newer) is present.
# The error reported gencode 7.35.1, so we need runtime >= 7.35.1.
%pip install --force-reinstall "protobuf>=7.35.1"

# 3. Install core dependencies, using --force-reinstall to override potential conflicts
%pip install --force-reinstall "pydantic>=2.0.0" "pydantic-settings" "chromadb>=0.4.0" "numpy>=2.0.0"

# 4. Install remaining dependencies
%pip install -U sentence-transformers transformers faiss-cpu

# Now, import the libraries after ensuring correct versions are installed
import numpy as np
import pandas as pd
import faiss
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer, InputExample
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from IPython.display import display

# Verify installed versions for debugging
print(f"ChromaDB version: {chromadb.__version__}")
import pydantic
print(f"Pydantic version: {pydantic.__version__}")
try:
    import pydantic_settings
    print("pydantic-settings is installed.")
except ImportError:
    print("pydantic-settings is NOT installed.")
import importlib.metadata
try:
    protobuf_version_after_all_installs = importlib.metadata.version('protobuf')
    print(f"Final Protobuf version detected: {protobuf_version_after_all_installs}")
except importlib.metadata.PackageNotFoundError:
    print("Protobuf not found after all installs.")

os.makedirs('cache', exist_ok=True)

Found existing installation: pydantic 2.13.4
Uninstalling pydantic-2.13.4:
  Successfully uninstalled pydantic-2.13.4
Found existing installation: pydantic-settings 2.14.2
Uninstalling pydantic-settings-2.14.2:
  Successfully uninstalled pydantic-settings-2.14.2
Found existing installation: chromadb 1.5.9
Uninstalling chromadb-1.5.9:
  Successfully uninstalled chromadb-1.5.9
Found existing installation: numpy 2.5.0
Uninstalling numpy-2.5.0:
  Successfully uninstalled numpy-2.5.0
Found existing installation: pydantic_core 2.46.4
Uninstalling pydantic_core-2.46.4:
  Successfully uninstalled pydantic_core-2.46.4
Found existing installation: protobuf 7.35.1
Uninstalling protobuf-7.35.1:
  Successfully uninstalled protobuf-7.35.1
Found existing installation: transformers 5.12.1
Uninstalling transformers-5.12.1:
  Successfully uninstalled transformers-5.12.1
Found existing installation: sentence-transformers 5.6.0
Uninstalling sentence-transformers-5.6.0:
  Successfully uninstalled sentence-

  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached pydantic_settings-2.14.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached chromadb-1.5.9-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.0 kB)
  Using cached numpy-2.5.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.46.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.6 kB)
  Using cached typing_extensions-4.16.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
  Using cached build-1.5.0-py3-none-any.whl.metadata (5.7 kB)
  Using cached pybase64-1.4.3-cp312-cp312-manylinux1_x86_64.manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_5_x86_64.whl.metadata (8.7 kB)
  Using cached uvicorn-0.49.0-py3-none-

  Using cached faiss_cpu-1.14.3-cp310-abi3-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (7.8 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.4/596.4 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 91.4 MB/s eta 0:00:00
Using cached faiss_cpu-1.14.3-cp310-abi3-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (18.5 MB)
Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.3 MB)
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.23.1
    Uninstalling tokenizers-0.23.1:
      Successfully uninstalled tokenizers-0.23.1
ChromaDB version: 1.5.9
Pydantic version: 2.13.4
pydantic-settings is installed.
Final Protobuf version detected: 7.35.1


## 🌟 Exercise 1 · Data loading and preparation

In [5]:
data_path = 'labelled_newscatcher_dataset.csv'
pdf = pd.read_csv(data_path, sep=';')
# Ensure 'id' column exists; if not, create it.
# The dataset already contains an 'id' column, so this block might not strictly be necessary but completes the TODO.
if 'id' not in pdf.columns:
    pdf['id'] = range(len(pdf))
display(pdf.head())
# Create a manageable subset (e.g., first 1000 rows) for easier processing.
pdf_subset = pdf.head(1000)
display(pdf_subset[['id', 'title']].head())

,topic,link,domain,published_date,title,lang,id
0,SCIENCE,https://www.eurekalert.org/pub_releases/2020-0...,eurekalert.org,2020-08-06 13:59:45,A closer look at water-splitting's solar fuel ...,en,0
1,SCIENCE,https://www.pulse.ng/news/world/an-irresistibl...,pulse.ng,2020-08-12 15:14:19,"An irresistible scent makes locusts swarm, stu...",en,1
2,SCIENCE,https://www.express.co.uk/news/science/1322607...,express.co.uk,2020-08-13 21:01:00,Artificial intelligence warning: AI will know ...,en,2
3,SCIENCE,https://www.ndtv.com/world-news/glaciers-could...,ndtv.com,2020-08-03 22:18:26,Glaciers Could Have Sculpted Mars Valleys: Study,en,3
4,SCIENCE,https://www.thesun.ie/tech/5742187/perseid-met...,thesun.ie,2020-08-12 19:54:36,Perseid meteor shower 2020: What time and how ...,en,4


,id,title
0,0,A closer look at water-splitting's solar fuel ...
1,1,"An irresistible scent makes locusts swarm, stu..."
2,2,Artificial intelligence warning: AI will know ...
3,3,Glaciers Could Have Sculpted Mars Valleys: Study
4,4,Perseid meteor shower 2020: What time and how ...


## 🌟 Exercise 2 · Vectorization with Sentence Transformers

In [8]:
def example_create_fn(idx: int, text: str) -> InputExample:
    return InputExample(guid=str(idx), texts=[text], label=0.0)

# Create training examples from the subset data using the example_create_fn
faiss_train_examples = [
    example_create_fn(row['id'], row['title'])
    for _, row in pdf_subset[['id', 'title']].iterrows()
]
faiss_train_examples[:2]

In [7]:
model = SentenceTransformer('all-MiniLM-L6-v2')
titles_list = pdf_subset['title'].tolist()
faiss_title_embedding = model.encode(titles_list, convert_to_numpy=True, show_progress_bar=True)
len(faiss_title_embedding), len(faiss_title_embedding[0])


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

(1000, 384)

## 🌟 Exercise 3 · FAISS indexing and search

In [9]:
pdf_to_index = pdf_subset
id_index = pdf_to_index['id'].to_numpy().astype(np.int64)
content_encoded_normalized = faiss_title_embedding.astype('float32')
faiss.normalize_L2(content_encoded_normalized)
index_content = faiss.IndexIDMap(faiss.IndexFlatIP(content_encoded_normalized.shape[1]))
index_content.add_with_ids(content_encoded_normalized, id_index)
index_content.ntotal


1000

In [12]:
def search_content(query: str, pdf_to_index: pd.DataFrame, k: int = 3):
    # Encode the query using the sentence transformer model
    query_vector = model.encode(query)
    # Reshape query_vector to be 2D, as faiss.normalize_L2 expects a 2D array (batch of vectors)
    query_vector = query_vector.reshape(1, -1)
    faiss.normalize_L2(query_vector)
    # Perform the search in the FAISS index
    sims, ids = index_content.search(query_vector, k)
    results = pdf_to_index[pdf_to_index['id'].isin(ids[0])].copy()
    results['similarities'] = sims[0]
    return results

display(search_content('animal', pdf_to_index, k=5))

,topic,link,domain,published_date,title,lang,id,similarities
99,TECHNOLOGY,https://www.gematsu.com/2020/08/ghostwire-toky...,gematsu.com,2020-08-07 16:43:13,Ghostwire: Tokyo confirms dog petting,en,99,0.391902
176,TECHNOLOGY,https://www.pushsquare.com/news/2020/08/random...,pushsquare.com,2020-08-03 16:30:00,Random: You Can Pick Up and Pet Cats in Assass...,en,176,0.376784
762,SCIENCE,https://af.reuters.com/article/worldNews/idAFK...,af.reuters.com,2020-08-13 16:51:00,'Secret' life of sharks: Study reveals their s...,en,762,0.344059
928,SCIENCE,https://www.thecut.com/2020/08/scientists-say-...,thecut.com,2020-08-04 12:52:00,Just Let This Lizard Be a Dinosaur,en,928,0.317387
975,HEALTH,https://www.news-medical.net/news/20200813/Res...,news-medical.net,2020-08-13 05:18:00,Researchers explore social behavior of animals...,en,975,0.295497


## 🌟 Exercise 4 · ChromaDB collection and querying

In [16]:
chroma_client = chromadb.Client(Settings(anonymized_telemetry=False))
collection_name = 'my_news'
if any(c.name == collection_name for c in chroma_client.list_collections()):
    chroma_client.delete_collection(name=collection_name)
# To-Do: create the collection and add documents
collection = chroma_client.create_collection(name=collection_name)

# Prepare data for adding to ChromaDB
# documents will be the titles
chroma_documents = titles_list

# embeddings will be the pre-calculated faiss_title_embedding
chroma_embeddings = faiss_title_embedding.tolist()

# metadatas can include other relevant columns from pdf_subset
# Ensure 'id' is not duplicated if it's already an explicit id in ChromaDB
chroma_metadatas = pdf_subset[['topic', 'link', 'domain', 'published_date']].to_dict('records')

# ids must be strings
chroma_ids = [str(x) for x in pdf_subset['id'].tolist()]

collection.add(
    documents=chroma_documents,
    embeddings=chroma_embeddings,
    metadatas=chroma_metadatas,
    ids=chroma_ids
)

# To-Do: query the collection
query_text = "animal"
results = collection.query(query_texts=[query_text], n_results=5)
print(json.dumps(results, indent=2))

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:00<00:00, 92.7MiB/s]


{
  "ids": [
    [
      "176",
      "975",
      "99",
      "928",
      "762"
    ]
  ],
  "embeddings": null,
  "documents": [
    [
      "Random: You Can Pick Up and Pet Cats in Assassin's Creed Valhalla",
      "Researchers explore social behavior of animals toward emerging infectious diseases",
      "Ghostwire: Tokyo confirms dog petting",
      "Just Let This Lizard Be a Dinosaur",
      "'Secret' life of sharks: Study reveals their surprising social networks"
    ]
  ],
  "uris": null,
  "included": [
    "metadatas",
    "documents",
    "distances"
  ],
  "data": null,
  "metadatas": [
    [
      {
        "link": "https://www.pushsquare.com/news/2020/08/random_you_can_pick_up_and_pet_cats_in_assassins_creed_valhalla",
        "published_date": "2020-08-03 16:30:00",
        "domain": "pushsquare.com",
        "topic": "TECHNOLOGY"
      },
      {
        "topic": "HEALTH",
        "domain": "news-medical.net",
        "link": "https://www.news-medical.net/news/20200813

## 🌟 Exercise 5 · Question answering with a Hugging Face model

In [19]:
model_id = 'google/flan-t5-small'  # lightweight, better than tiny GPT-2 for QA
# To-Do: create the text2text-generation pipeline
pipe = pipeline("text-generation", model=model_id)

question = "What's the latest news on space development?"
context_docs = results['documents'][0][:3]
context = ' '.join(context_docs)
prompt = f"Answer the question using only the context.\nContext: {context}\nQuestion: {question}\nAnswer:\n"
response = pipe(prompt)[0]['generated_text']
print(response)

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DeepseekV32ForCausalLM', 'DeepseekV4ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCaus

Answer the question using only the context.
Context: Random: You Can Pick Up and Pet Cats in Assassin's Creed Valhalla Researchers explore social behavior of animals toward emerging infectious diseases Ghostwire: Tokyo confirms dog petting
Question: What's the latest news on space development?
Answer:

